In [1]:
import os
import time
import random
import numpy as np
from datetime import datetime
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm  
from torchinfo import summary

# --- Project-specific imports ---
import config_potamogaton as cfg
import cnn_utils.data as data_utils
from cnn_utils.model import create_model_from_config, save_model_weights
from cnn_utils.evaluate import EvalClassification

torch.manual_seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
random.seed(cfg.random_seed)

print("Imports complete. Ready to set up the training run.\n")

train_dataloader, val_dataloader, test_dataloader = data_utils.get_3_dataloaders(cfg)
val_ood_dataloader, test_ood_dataloader = None, None

2025-09-18 14:00:42.450252: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-18 14:00:42.461012: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758184242.473380   12899 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758184242.477035   12899 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758184242.486294   12899 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Imports complete. Ready to set up the training run.

 Total dataset size 	: 105
 Train dataset size 	: 83
 Val dataset size 	: 11
 Test dataset size 	: 11



## Load model

In [2]:
model = create_model_from_config(cfg)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
print("Number of trainable parameters: ", sum(p.numel() for p in trainable_params))

summary(model, input_size=(1, 3, 224, 224))

Number of trainable parameters:  5383


Layer (type:depth-idx)                                            Output Shape              Param #
LinearBaselineModel                                               [1, 7]                    --
├─ConvNeXt: 1-1                                                   [1, 768]                  --
│    └─Sequential: 2-1                                            [1, 96, 56, 56]           --
│    │    └─Conv2d: 3-1                                           [1, 96, 56, 56]           (4,704)
│    │    └─LayerNorm2d: 3-2                                      [1, 96, 56, 56]           (192)
│    └─Sequential: 2-2                                            [1, 768, 7, 7]            --
│    │    └─ConvNeXtStage: 3-3                                    [1, 96, 56, 56]           (239,904)
│    │    └─ConvNeXtStage: 3-4                                    [1, 192, 28, 28]          (996,288)
│    │    └─ConvNeXtStage: 3-5                                    [1, 384, 14, 14]          (11,137,152)
│    │    └─C

## Training Loop
- Tensorboard logging
- Save model checkpoints
- Save best models

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device != torch.device("cuda"):
    raise RuntimeError("CUDA is not available. It is preffered to run this script on a GPU for training.")
    # You can comment out this line if you want to run on CPU for testing purposes.
    
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.to(device)
print(f"Using device: {device}, {torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU'}")

Using device: cuda, NVIDIA RTX A500 Laptop GPU


In [6]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))
optimizer = torch.optim.AdamW(trainable_params, lr=cfg.lr_initial, weight_decay=cfg.weight_decay)

if cfg.lr_scheme == 'cosine':
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.lr_cosine_tmax_epochs, eta_min=cfg.lr_cosine_minimum)
elif cfg.lr_scheme == 'step':
    scheduler = StepLR(optimizer, step_size=cfg.lr_step_size, gamma=cfg.lr_step_gamma)
else:
    print(f"Unsupported learning rate scheme: {cfg.lr_scheme}. Using Fixed LR.")
    scheduler = None

now = datetime.now()
timestamp = now.strftime("%m-%d_%H-%M--%S")

# Save a copy of the configuration file
config_file_name = f"config_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.py"
cfg.save_this_config(config_file_name)

hparams_dict = {
    "a_run_name": cfg.training_run_name,
    "data_directory": cfg.main_data_dir,
    "train_dataset_percent": cfg.train_percent,
    "val_dataset_percent": cfg.val_percent,
    "test_dataset_percent": cfg.test_percent,
    "model_architecture": cfg.train_timm_model_name,
    "classification_head": cfg.classification_head_type,
    "num_trainable_parameters": sum(p.numel() for p in trainable_params),
    "optimizer": optimizer.__class__.__name__,
    "criterion": criterion.__class__.__name__,
    "scheduler": scheduler.__class__.__name__,
    "num_epochs": cfg.num_epochs,
    "batch_size": cfg.batch_size,
    "learning_rate_scheme": cfg.lr_scheme,
    "initial_learning_rate": cfg.lr_initial,
    "final_learning_rate": cfg.lr_cosine_minimum,
    "weight_decay": cfg.weight_decay,
    "random_seed": cfg.random_seed,
    "timestamp": timestamp,
    "config_file_path": os.path.join(cfg.saved_configs_path, config_file_name)
}

writer = SummaryWriter(log_dir=os.path.join(cfg.base_logs_path, f"{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}"))

for key, value in hparams_dict.items():
    writer.add_text(key, str(value))

completed_all_epochs = False

# Training loop
best_val_loss = float('inf') 
best_loss_epoch = 0
best_val_accuracy = 0.0
best_acc_epoch = 0

if cfg.ood:
    best_val_ood_loss = float('inf')
    best_ood_loss_epoch = 0
    best_val_ood_accuracy = 0.0
    best_ood_acc_epoch = 0

patience_counter_loss = 0
patience_counter_acc = 0

print("Starting training...")
start_time = time.time()
for epoch in range(cfg.num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{cfg.num_epochs}", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / total
    train_accuracy = correct / total
    
    lr = scheduler.get_last_lr()[0] if scheduler else cfg.lr_initial
    print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy*100:.4f}% at LR: {lr:.6f}")
    
    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_accuracy, epoch)
    
    # Validation phase
    if (epoch + 1) % cfg.validation_interval == 0:
        model.eval()  
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        val_ood_running_loss = 0.0
        val_ood_correct = 0
        val_ood_total = 0
        
        with torch.no_grad():  
            
            # validation
            for val_images, val_labels in val_dataloader:
                val_images, val_labels = val_images.to(device), val_labels.to(device)
                
                val_outputs = model(val_images)
                val_loss = criterion(val_outputs, val_labels)
                
                val_running_loss += val_loss.item() * val_images.size(0)
                _, val_predicted = torch.max(val_outputs.data, 1)
                val_total += val_labels.size(0)
                val_correct += (val_predicted == val_labels).sum().item()
        # val loss and acc
        val_loss = val_running_loss / val_total
        val_accuracy = val_correct / val_total

        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy*100:.4f}%")
        
        writer.add_scalar('Loss/validation', val_loss, epoch)
        writer.add_scalar('Accuracy/validation', val_accuracy, epoch)
        
        if cfg.ood:
            with torch.no_grad():
                # out of distribution validation
                for val_images, val_labels in val_ood_dataloader:
                    val_images, val_labels = val_images.to(device), val_labels.to(device)
                    
                    val_ood_outputs = model(val_images)
                    val_ood_loss = criterion(val_ood_outputs, val_labels)
                    
                    val_ood_running_loss += val_ood_loss.item() * val_images.size(0)
                    _, val_ood_predicted = torch.max(val_ood_outputs.data, 1)
                    val_ood_total += val_labels.size(0)
                    val_ood_correct += (val_ood_predicted == val_labels).sum().item()
        
            # OOD validation
            val_ood_loss = val_ood_running_loss / val_ood_total
            val_ood_accuracy = val_ood_correct / val_ood_total
            
            print(f"Validation OOD Loss: {val_ood_loss:.4f}, Validation OOD Accuracy: {val_ood_accuracy*100:.4f}%")
            
            writer.add_scalar('Loss/validation_ood', val_ood_loss, epoch)
            writer.add_scalar('Accuracy/validation_ood', val_ood_accuracy, epoch)
     
     
            # BEST MODEL SELECTION
            # For now, we are saving all promising models
            #   - best losses and accuracies for val and ood sets
            # That will make 4 + 1 models saved at the end of training for each run.
            # The Patience thing is only with the in-domain validation set - and early stopping is disabled for now.
            # Of these, the most promising should be the one with best ood val loss or maybe accuracy - is what i think.
            
            #  this part is still if cfg.ood is True.
        
            if val_ood_loss < best_val_ood_loss:
                best_val_ood_loss = val_ood_loss
                best_ood_loss_epoch = epoch + 1
                
                best_ood_model_name = f"best_ood_loss_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
                best_ood_model_path = os.path.join(cfg.models_path, best_ood_model_name)
                if os.path.exists(best_ood_model_path):
                    os.remove(best_ood_model_path)
                    
                save_model_weights(model, best_ood_model_path, verbose=False)
                print(f"Saved new best OOD loss model. Val OOD Loss: {best_val_ood_loss:.4f} at Epoch {best_ood_loss_epoch}")
                
                writer.add_scalar('Best_Validation_OOD_Loss', best_val_ood_loss, epoch)
                
            if val_ood_accuracy > best_val_ood_accuracy:
                best_val_ood_accuracy = val_ood_accuracy
                best_ood_acc_epoch = epoch + 1
                
                best_ood_acc_model_name = f"best_ood_acc_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
                best_ood_acc_model_path = os.path.join(cfg.models_path, best_ood_acc_model_name)
                if os.path.exists(best_ood_acc_model_path):
                    os.remove(best_ood_acc_model_path)
                    
                save_model_weights(model, best_ood_acc_model_path, verbose=False)
                print(f"Saved new best OOD accuracy model. Val OOD Acc: {best_val_ood_accuracy*100:.4f}% at Epoch {best_ood_acc_epoch}")
                
                writer.add_scalar('Best_Validation_OOD_Accuracy', best_val_ood_accuracy, epoch)
        
        # this part is always executed, regardless of cfg.ood
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_loss_epoch = epoch + 1
            patience_counter_loss = 0

            best_loss_model_name = f"best_loss_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
            best_loss_model_path = os.path.join(cfg.models_path, best_loss_model_name)
            
            if os.path.exists(best_loss_model_path):
                os.remove(best_loss_model_path)
            
            save_model_weights(model, best_loss_model_path, verbose=False)
            print(f"Saved new best loss model. Val Loss: {best_val_loss:.4f} at Epoch {best_loss_epoch}")
            writer.add_scalar('Best_Validation_Loss', best_val_loss, epoch)
        else:
            patience_counter_loss += 1

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_acc_epoch = epoch + 1
            patience_counter_acc = 0

            best_acc_model_name = f"best_acc_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
            best_acc_model_path = os.path.join(cfg.models_path, best_acc_model_name)
            
            if os.path.exists(best_acc_model_path):
                os.remove(best_acc_model_path)
                
            save_model_weights(model, best_acc_model_path, verbose=False)
            print(f"Saved new best accuracy model. Val Acc: {best_val_accuracy*100:.4f}% at Epoch {best_acc_epoch}")
            writer.add_scalar('Best_Validation_Accuracy', best_val_accuracy, epoch)
        else:
            patience_counter_acc += 1
        
        # Early stopping based on patience
        if cfg.early_stopping:
            if patience_counter_loss >= cfg.patience_loss and patience_counter_acc >= cfg.patience_accuracy:
                print(f"\n\nEarly stopping would be triggered at epoch {epoch + 1}.\n\n")
                early_stop_path_name = f"early_stop_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
                early_stop_model_path = os.path.join(cfg.models_path, early_stop_path_name)
                save_model_weights(model, early_stop_model_path, verbose=False)
                break # Uncomment this line to enable early stopping
            
    
    # Update learning rate
    if scheduler:
        scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Learning_Rate', current_lr, epoch)
    
    # Checkpoint saving phase
    # if (epoch + 1) % cfg.checkpoint_interval == 0 or epoch == cfg.num_epochs - 1:
    #     checkpoint_name = f"checkpoint_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}_epoch_{epoch + 1}.pth"
    #     checkpoint_file_path = os.path.join(cfg.checkpoint_path, checkpoint_name)
        
    #     torch.save({
    #         'epoch': epoch + 1,
    #         'model_config': model.config,
    #         'model_state_dict': model.state_dict(),
    #         'optimizer_state_dict': optimizer.state_dict(),
    #         'scheduler_state_dict': scheduler.state_dict(),
    #         'train_loss': train_loss, 
    #         'log_dir': writer.log_dir,
    #         'best_val_accuracy': best_val_accuracy,
    #         'best_val_loss': best_val_loss,
    #         'best_val_acc_epoch': best_acc_epoch,
    #         'best_val_loss_epoch': best_loss_epoch,
    #         'hparams': hparams_dict
    #     }, checkpoint_file_path)
        
    #     print(f"Saved checkpoint: {checkpoint_file_path}")
        
        # checkpoint_model_name = f"checkpoint_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}_epoch_{epoch + 1}.pth"
        # checkpoint_model_path = os.path.join(cfg.models_path, checkpoint_model_name)
        # save_model_weights(model, checkpoint_model_path, verbose=False)
        
        # print(f"Saved model checkpoint: {checkpoint_model_path}")
        
print("\nTraining complete.")

print(f"\nBest Validation Accuracy: {best_val_accuracy*100:.4f}% at Epoch {best_acc_epoch}")
print(f"Best Validation Loss: {best_val_loss:.4f} at Epoch {best_loss_epoch}")

if cfg.ood:
    print(f"Best Validation OOD Accuracy: {best_val_ood_accuracy*100:.4f}% at Epoch {best_ood_acc_epoch}")
    print(f"Best Validation OOD Loss: {best_val_ood_loss:.4f} at Epoch {best_ood_loss_epoch}")

print(f"\nFinal Training Loss (at end of last epoch): {train_loss:.4f}")
print(f"Final Training Accuracy (at end of last epoch): {train_accuracy*100:.4f}%")

completed_all_epochs = (epoch + 1 == cfg.num_epochs)
end_time = time.time()
elapsed_time = end_time - start_time
elapsed_time_hms = time.strftime("%H:%M:%S", time.gmtime(elapsed_time))
print(f"Total Training Time: {elapsed_time_hms}")
print(f"{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}")
  
# Save the final model
final_model_name = f"final_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
final_model_file_path = os.path.join(cfg.models_path, final_model_name)
save_model_weights(model, final_model_file_path)

Configuration saved as: config_09-18_14-01--17_a1_potamogaton_convnextv2_tiny.fcmae_ft_in22k_in1k.py in /home/takayuki/Desktop/summer2025/plants/plant_classification/../saved_configs
Starting training...


Epoch 1/30:   0%|          | 0/21 [00:00<?, ?batch/s]

Epoch 1/30: 100%|██████████| 21/21 [00:11<00:00,  1.86batch/s]


Training Loss: 1.5957, Training Accuracy: 32.5301% at LR: 0.000500
Validation Loss: 1.1854, Validation Accuracy: 72.7273%
Saved new best loss model. Val Loss: 1.1854 at Epoch 1
Saved new best accuracy model. Val Acc: 72.7273% at Epoch 1


Epoch 2/30:  38%|███▊      | 8/21 [00:04<00:07,  1.68batch/s]


KeyboardInterrupt: 

In [7]:
if not completed_all_epochs:
    print("Training was interrupted before completing all epochs.")
    end_time = time.time()
    elapsed_time = end_time - start_time
    # Save the final model
    final_model_name = f"final_model_{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}.pth"
    final_model_file_path = os.path.join(cfg.models_path, final_model_name)
    save_model_weights(model, final_model_file_path)

print("\nTraining complete.")

print(f"\nBest Validation Accuracy: {best_val_accuracy*100:.4f}% at Epoch {best_acc_epoch}")
print(f"Best Accuracy Model saved as: {best_acc_model_name}")

print(f"\nBest Validation Loss: {best_val_loss:.4f} at Epoch {best_loss_epoch}")
print(f"Best Loss Model saved as: {best_loss_model_name}")

if cfg.ood:
    print(f"\nBest Validation OOD Accuracy: {best_val_ood_accuracy*100:.4f}% at Epoch {best_ood_acc_epoch}")
    print(f"Best OOD Accuracy Model saved as: {best_ood_acc_model_name}")

    print(f"\nBest Validation OOD Loss: {best_val_ood_loss:.4f} at Epoch {best_ood_loss_epoch}")
    print(f"Best OOD Loss Model saved as: {best_ood_model_name}")

print(f"\nFinal Training Loss: {train_loss:.4f}")
print(f"Final Training Accuracy: {train_accuracy*100:.4f}%")

elapsed_time_hms = time.strftime("%H:%M:%S", time.gmtime(elapsed_time))
print(f"Total Training Time: {elapsed_time_hms}")

print("\nRun ID: ")
print(f"{timestamp}_{cfg.training_run_name}_{cfg.train_timm_model_name}")

Training was interrupted before completing all epochs.
Model weights and config saved to /home/takayuki/Desktop/summer2025/plants/plant_classification/../training/potamogaton/models/final_model_09-18_14-01--17_a1_potamogaton_convnextv2_tiny.fcmae_ft_in22k_in1k.pth

Training complete.

Best Validation Accuracy: 72.7273% at Epoch 1
Best Accuracy Model saved as: best_acc_model_09-18_14-01--17_a1_potamogaton_convnextv2_tiny.fcmae_ft_in22k_in1k.pth

Best Validation Loss: 1.1854 at Epoch 1
Best Loss Model saved as: best_loss_model_09-18_14-01--17_a1_potamogaton_convnextv2_tiny.fcmae_ft_in22k_in1k.pth

Final Training Loss: 1.5957
Final Training Accuracy: 32.5301%
Total Training Time: 00:00:21

Run ID: 
09-18_14-01--17_a1_potamogaton_convnextv2_tiny.fcmae_ft_in22k_in1k


## Evaluation script

#### Some evaluation metrics
- Accuracy: correct predictions / total predictions
- Precision: TP / (TP + FP)
- Recall: TP / (TP + FN)
- F1 Score: Harmonic mean of precision and recall

In [8]:
# In a new cell at the end of your training notebook

from cnn_utils.evaluate import EvalClassification
from cnn_utils.model import load_model 

eval = EvalClassification(cfg, model, test_dataloader)
eval.evaluate()
eval.get_accuracy()

  -- Evaluating: 100%|██████████| 11/11 [00:01<00:00,  6.91it/s]

Accuracy: 0.7273


0.7272727272727273

### Hyperparameters get logged to tensorboard only if the following block is executed after noting observations

In [9]:
observations = "this was a refactoring test" 

hparams_dict["observations"] = observations
writer.add_text('Observations', observations, 0)

# 2. Calculate total elapsed time
elapsed_time_minutes = (time.time() - start_time) / 60

final_metrics_to_log = {
    
    "indicators/best_val_accuracy": best_val_accuracy,
    "indicators/best_val_loss": best_val_loss,
    "indicators/best_val_epoch": best_loss_epoch,
    "indicators/best_acc_epoch": best_acc_epoch,
    
    "indicators/final_val_accuracy": val_accuracy,
    "indicators/final_train_accuracy": train_accuracy,
    "indicators/final_train_loss": train_loss,
    "indicators/final_val_loss": val_loss,
    "indicators/final_learning_rate": current_lr,
    
    "meta/epochs_trained": epoch + 1,
    "meta/time_elapsed_minutes": elapsed_time_minutes
}

# 4. Write to TensorBoard and close the writer
# Pass the pure hparams and the pure metrics separately.
writer.add_hparams(hparams_dict, final_metrics_to_log)
writer.flush()
writer.close()

print("\nAll metrics and hyperparameters logged correctly to TensorBoard.")


All metrics and hyperparameters logged correctly to TensorBoard.
